<a href="https://colab.research.google.com/github/mdtokarev/App-Stat-Lab01/blob/main/AppStatLab01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab01 — sprint 1**

Лабораторная работа № 1 по прикладной статистике.

**Авторы:**   
**Группа:** ПрикСтат Инф25 1.3

## Цель исследования

Исследовать три случайные величины на химических данных,
описать их распределения и зависимости, оценить точность
выборочных характеристик и проверить статистические гипотезы.

## Текущий статус

Подготовлена рабочая среда.
Выбран датасет экспериментальных оптических свойств хромофоров. После выполнения отбора данных выборка насчитывает 1968 записей в дихлорметане.


## Среда выполнения

В следующей ячейке импортируем библиотеки для дальнейшей работы.
Фиксированное начальное состояние генератора используется
для воспроизводимости последующих случайных выборок.

In [10]:
import sys

import numpy as np # численные расчёты и случайные выборки
import pandas as pd # работа с таблицами
import scipy # распределения и статичтические методы
import matplotlib
import matplotlib.pyplot as plt # графики

SEED = 67 # воспроизводимость случайных операций (получение одинаковой выборки и результатов)
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Готов к труду и обороне!")

Python: 3.13.18
NumPy: 2.1.3
pandas: 2.2.3
SciPy: 1.16.3
Matplotlib: 3.10.0
Готов к труду и обороне!


## Данные и постановка исследования

Исследуются экспериментальные оптические свойства органических
соединений в дихлорметане.

Источник данных:
https://doi.org/10.6084/m9.figshare.12045567

Статья с описанием:
https://doi.org/10.1038/s41597-020-00634-8

Используется файл `DB for chromophore_Sci_Data_rev03.csv`.

Выбранные величины:
1. Максимум поглощения — `Absorption max (nm)`, нм.
2. Максимум испускания — `Emission max (nm)`, нм.
3. Квантовый выход — `Quantum yield`, безразмерная величина.

Исследовательский вопрос:
как распределены эти показатели в выбранном наборе хромофоров
и какие связи наблюдаются между ними?

Для уменьшения различий условий рассматриваются записи
в одном растворителе — дихлорметане (`ClCCl`).

In [11]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "mdtokarev/App-Stat-Lab01/main/"
    "DB%20for%20chromophore_Sci_Data_rev03.csv"
)

raw = pd.read_csv(DATA_URL)

print("Число строк:", raw.shape[0])
print("Число столбцов:", raw.shape[1])

display(raw[
    [
        "Tag",
        "Solvent",
        "Absorption max (nm)",
        "Emission max (nm)",
        "Quantum yield",
    ]
].head(10))

Число строк: 20836
Число столбцов: 14


,Tag,Solvent,Absorption max (nm),Emission max (nm),Quantum yield
0,0,O,355.0,410.0,NaN
1,1,O,408.0,450.0,NaN
2,2,ClC(Cl)Cl,526.0,535.0,NaN
3,3,CC#N,514.0,553.7,NaN
4,4,CS(C)=O,524.0,555.0,NaN
5,5,C1CCCCC1,NaN,451.0,NaN
6,6,C1CCCCC1,NaN,639.0,NaN
7,7,C1CCCCC1,350.0,395.0,NaN
8,8,CCOC(C)=O,361.0,416.0,NaN
9,9,CC#N,367.0,430.0,NaN


## Формирование исследуемого набора данных

Применяются следующие правила:

1. Оставить записи, в которых заполнены все три выбранных показателя.
2. Удалить точные дублирующиеся записи в таблице.
3. Оставить в качестве растворителя только дихлорметан: значение Solvent равно ClCCl.
4. Выявить повторяющиеся хромофоры и проверить их данные.

Повторение хромофора не обязательно будет означать ошибку - записи могут соответствовать разным измерениям.

Таким образом, исследуем набор измерений, отобранных по растворителю и полноте трёх показателей. Такой набор не будет являться случайной выборкой всех органических молекул.

In [12]:
# Три показателя, которые будем исследовать
TARGETS = [
    "Absorption max (nm)",
    "Emission max (nm)",
    "Quantum yield",
]

# 1. Оставляем строки с заполненными тремя показателями
complete = raw.dropna(subset=TARGETS).copy()

# 2. Удаляем точные дубли
comparison_columns = [
    column for column in raw.columns
    if column != "Tag" # Tag - служебный номер, поэтому его не учитываем
]

deduplicated = complete.drop_duplicates(
    subset=comparison_columns
).copy()

# 3. Оставляем только измерения в дихлорметане
data = deduplicated.loc[
    deduplicated["Solvent"] == "ClCCl"
].copy()

# Показываем, сколько строк осталось после каждого шага
cleaning_summary = pd.DataFrame({
    "Этап": [
        "Исходная таблица",
        "Заполнены три выбранных показателя",
        "Удалены точные дубли",
        "Оставлен только дихлорметан",
    ],
    "Осталось строк": [
        len(raw),
        len(complete),
        len(deduplicated),
        len(data),
    ],
})

display(cleaning_summary)

,Этап,Осталось строк
0,Исходная таблица,20836
1,Заполнены три выбранных показателя,12211
2,Удалены точные дубли,12183
3,Оставлен только дихлорметан,1975


После отбора осталось 1975 записей в дихлорметане
с заполненными максимумами поглощения, испускания
и квантовым выходом.

Пропуски в других столбцах (помимо трёх исследуемых) не учитываем как исключающий параметр.

Повторные записи одного хромофора с различающимися данными
пока сохраняем.

На этом этапе выполнен первоначальный отбор данных.

## Проверка повторных записей хромофоров

Выделим хромофоры, которые встречаются в рабочем наборе
более одного раза. Сравним их оптические показатели
и ссылки на исходные публикации.

Повторение определяем по точному совпадению строки Chromophore.

In [13]:
# keep=False отмечает ВСЕ строки повторяющихся хромофоров
repeated_mask = data.duplicated(
    subset="Chromophore",
    keep=False,
)

repeated_data = data.loc[repeated_mask].copy()

print(
    "Хромофоров с несколькими записями:",
    repeated_data["Chromophore"].nunique(),
)
print("Всего записей этих хромофоров:", len(repeated_data))

# Присваиваем группам короткие номера для удобства просмотра
repeated_data["Группа"] = (
    repeated_data.groupby("Chromophore", sort=True).ngroup() + 1
)

display(
    repeated_data.sort_values(["Группа", "Tag"])[
        ["Группа", "Tag", *TARGETS, "Reference"]
    ]
)

Хромофоров с несколькими записями: 23
Всего записей этих хромофоров: 46


,Группа,Tag,Absorption max (nm),Emission max (nm),Quantum yield,Reference
13960,1,13960,538.0,560.0,0.600,10.1016/j.jphotobiol.2017.12.006
20408,1,20408,538.0,560.0,0.600,10.1021/ol503379c
13956,2,13956,463.0,552.0,0.720,10.1016/j.jphotobiol.2017.12.006
20430,2,20430,494.0,523.0,0.710,10.1021/ol503379c
13959,3,13959,538.0,543.0,0.700,10.1016/j.jphotobiol.2017.12.006
20403,3,20403,524.0,543.0,0.700,10.1021/ol503379c
13955,4,13955,451.0,534.0,0.860,10.1016/j.jphotobiol.2017.12.006
20429,4,20429,482.0,505.0,0.860,10.1021/ol503379c
10641,5,10641,518.0,528.0,0.001,10.1016/j.dyepig.2015.11.028
17082,5,17082,518.0,528.0,0.009,10.1039/c7cp08472b


В датасете обнаружились строки с полностью совпадающими исследуемыми параметрами - удаляем их (для одного хромофора в одном и том же растворителе).

In [14]:
# Сохраняем набор до этого шага для сравнения
data_before_dedup = data.copy()

# Поля, по которым определяем совпадение для нашего анализа
analysis_key = ["Chromophore", "Solvent", *TARGETS]

# Собираем все ссылки для каждой комбинации значений
source_links = (
    data_before_dedup
    .groupby(analysis_key, dropna=False)["Reference"]
    .agg(
        lambda refs: "; ".join(
            sorted(set(refs.dropna().astype(str)))
        )
    )
    .rename("All_references")
    .reset_index()
)

# Оставляем одну строку на комбинацию и присоединяем все её источники
data = (
    data_before_dedup
    .drop_duplicates(subset=analysis_key)
    .merge(
        source_links,
        on=analysis_key,
        how="left",
        validate="one_to_one",
    )
)

print("До:", len(data_before_dedup))
print("После:", len(data))
print("Удалено лишних копий:", len(data_before_dedup) - len(data))

До: 1975
После: 1968
Удалено лишних копий: 7


## Первичная проверка качества данных

Параметры оценки - отсутствуют пропуски в исследуемых показателях, максимумы поглощения и испускания положительны, квантовый выход лежит от 0 до 1. Отдельно рассматриваем ситуацию, когда максимум испускания меньше максимума поглощения.

In [15]:
quality_checks = pd.Series({
    "Пропуски в трёх показателях":
        int(data[TARGETS].isna().sum().sum()),

    "Максимум поглощения ≤ 0":
        int((data["Absorption max (nm)"] <= 0).sum()),

    "Максимум испускания ≤ 0":
        int((data["Emission max (nm)"] <= 0).sum()),

    "Квантовый выход вне диапазона [0, 1]":
        int((~data["Quantum yield"].between(0, 1)).sum()),

    "Максимум испускания меньше максимума поглощения":
        int((
            data["Emission max (nm)"]
            < data["Absorption max (nm)"]
        ).sum()),
}, name="Количество")

display(quality_checks.to_frame())

,Количество
Пропуски в трёх показателях,0
Максимум поглощения ≤ 0,0
Максимум испускания ≤ 0,0
"Квантовый выход вне диапазона [0, 1]",0
Максимум испускания меньше максимума поглощения,2


## Результаты первичной проверки качества

В выбранных трёх показателях нет пропусков.
Длины волн положительны, квантовый выход находится
в диапазоне [0, 1].

Обнаружены две записи, в которых максимум испускания
меньше максимума поглощения. Отдельно проверим эти записи.

In [16]:
# Выбираем записи, требующие проверки
suspect_mask = (
    data["Emission max (nm)"]
    < data["Absorption max (nm)"]
)

suspect_data = data.loc[suspect_mask].copy()

# Разность длин волн поможет оценить величину расхождения
suspect_data["Разность длин волн, нм"] = (
    suspect_data["Emission max (nm)"]
    - suspect_data["Absorption max (nm)"]
)

display(
    suspect_data[
        [
            "Tag",
            "Chromophore",
            *TARGETS,
            "Разность длин волн, нм",
            "All_references",
        ]
    ]
)

,Tag,Chromophore,Absorption max (nm),Emission max (nm),Quantum yield,"Разность длин волн, нм",All_references
246,2601,CN(C)CCN(Cc1c2ccccc2cc2ccccc12)c1ccc([N+](=O)[...,394.0,393.0,0.006,-1.0,10.1021/jp047557b
483,5084,CC1=C(c2ccc(N(C)C)cc2)C(C)=[N+]2C1=C(c1c(-c3cc...,538.0,524.0,0.004,-14.0,10.1016/j.dyepig.2018.04.070


## Проверка отмеченных записей по первоисточникам

**Tag 2601.** В статье Bag и Bharadwaj (2005),
таблица 3, с. 4386 (страница 10 PDF), для соединения 2
в дихлорметане указаны максимумы поглощения 394 нм
и испускания 393 нм. Квантовый выход 0,0057 соответствует
округлённому значению 0,006 в датасете.
Запись сохраняется.

Источник: https://doi.org/10.1021/jp047557b

**Tag 5084.** В статье Zhang и соавторов (2018),
таблица 3 (страница 17  PDF), для соединения 9 в дихлорметане
указаны поглощение 538 нм и два максимума испускания: 524 и 572 нм.
Значение 524 нм также обсуждается в тексте статьи.
В датасете представлен этот максимум.
Квантовый выход 0,41% = 0,0041 соответствует округлению
до 0,004 в датасете.

Запись сохраняется с учётом того, что одна величина
не отражает всю структуру спектра. Указанный в статье
стоксов сдвиг 1105 см⁻¹ соответствует паре 538 и 572 нм.

Источник: https://doi.org/10.1016/j.dyepig.2018.04.070

Обе записи оставляем без изменения.
Размер рабочей выборки данных — 1968 записей.